# Trực quan hóa bộ dữ liệu Sentinel-2 + RPG (PASTIS24 tự thu)

Hai mùa: **2019-2020** (nhãn RPG 2020) và **2020-2021** (nhãn RPG 2021). Mỗi mẫu là một cửa sổ 24×24 pixel (240 m × 240 m), 8–12 ngày chụp thật, 10 băng phổ, nhãn semantic `labels[0]` (19 = void).

Cách dùng: **Add Input** → chọn dataset chứa `pastis24_2019_2021.zip`. Notebook tự tìm `collected_pastis24_*.zip` (đọc pickle trực tiếp trong ZIP, không giải nén) hoặc thư mục đã giải nén có `sample_paths.csv`. Không cần GPU hay Internet.

In [ ]:
import io, json, pickle, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

DATA_ROOT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path("outputs")
SEED = 0
SAMPLES_PER_SEASON = 4      # số mẫu hiển thị ảnh/nhãn mỗi mùa
PROFILE_SAMPLES = 400       # số mẫu dùng tính đường NDVI theo lớp
CLASS_NAMES = ["Background", "Meadow", "Soft winter wheat", "Corn", "Winter barley",
               "Winter rapeseed", "Spring barley", "Sunflower", "Grapevine", "Beet",
               "Winter triticale", "Winter durum wheat", "Fruits, vegetables, flowers",
               "Potatoes", "Leguminous fodder", "Soybeans", "Orchard", "Mixed cereal",
               "Sorghum", "Void"]
VOID = 19
BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
colors = plt.cm.tab20(np.linspace(0, 1, 20))
colors[VOID] = [0.75, 0.75, 0.75, 1]
LABEL_CMAP = ListedColormap(colors)


def find_seasons(root):
    """season -> read(name) returning bytes, from a collected ZIP or an extracted folder."""
    seasons = {}
    for path in sorted(root.rglob("collected_pastis24_*.zip")):
        seasons[path.stem.removeprefix("collected_pastis24_")] = zipfile.ZipFile(path).read
    for paths_csv in sorted(root.rglob("sample_paths.csv")):
        folder = paths_csv.parent
        name = folder.name.removeprefix("collected_pastis24_")
        seasons.setdefault(name, lambda n, d=folder: (d / n).read_bytes())
    if not seasons:
        raise FileNotFoundError(f"No collected_pastis24_*.zip or sample_paths.csv under {root}")
    return seasons


SEASONS = find_seasons(DATA_ROOT)
print("Seasons:", list(SEASONS))


def load_sample(season, path):
    # Pickles come from this project's own collector; do not load untrusted files.
    return pickle.loads(SEASONS[season](path))

## 1. Tổng quan số mẫu

In [ ]:
frames, reports, verifications = [], {}, {}
for season, read in SEASONS.items():
    frame = pd.read_csv(io.BytesIO(read("manifest.csv")))
    frame["season"] = season
    frames.append(frame)
    reports[season] = json.loads(read("report.json"))
    verifications[season] = json.loads(read("verification.json"))
df = pd.concat(frames, ignore_index=True)
df["date_list"] = df["dates"].str.split(";")

print("Tổng số mẫu:", len(df))
display(df.groupby(["season", "region"]).size().unstack(fill_value=0).assign(total=lambda x: x.sum(axis=1)))
display(pd.DataFrame({s: {"collected": r["collected_samples"], "target_met": r["target_met"],
                          "sample_GB": round(r["sample_bytes"] / 1e9, 2),
                          "verified": verifications[s]["verified_samples"]}
                      for s, r in reports.items()}).T)

## 2. Phân bố số ngày chụp, độ trong (SCL) và số pixel có nhãn

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for season, part in df.groupby("season"):
    axes[0, 1].hist(part["clear_fraction"], bins=30, alpha=0.6, label=season)
    axes[1, 0].hist(part["known_label_pixels"], bins=30, alpha=0.6, label=season)
    months = pd.to_datetime(part["date_list"].explode()).dt.to_period("M").value_counts().sort_index()
    axes[1, 1].plot(months.index.astype(str), months.values, marker="o", label=season)
pd.crosstab(df["observations"], df["season"]).plot.bar(ax=axes[0, 0], rot=0)
axes[0, 0].set(title="Số ngày chụp mỗi mẫu", xlabel="T", ylabel="số mẫu")
axes[0, 1].set(title="Tỉ lệ pixel trong (SCL 4/5/6)", xlabel="clear_fraction")
axes[1, 0].set(title="Pixel có nhãn (≠ void) / 576", xlabel="known_label_pixels")
axes[1, 1].set(title="Số ảnh theo tháng (mỗi mẫu tính 1 lần/ngày)", ylabel="lượt")
axes[1, 1].tick_params(axis="x", rotation=60)
for ax in axes.flat:
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Vị trí các cửa sổ
Mỗi vùng một hệ UTM riêng nên vẽ tách từng vùng. Mỗi chấm là góc tây-bắc một cửa sổ 240 m.

In [ ]:
regions = sorted(df["region"].unique())
fig, axes = plt.subplots(1, len(regions), figsize=(4.5 * len(regions), 4.5), squeeze=False)
for ax, region in zip(axes[0], regions):
    for season, part in df[df["region"] == region].groupby("season"):
        ax.scatter(part["west"] / 1000, part["north"] / 1000, s=1, alpha=0.5, label=season)
    ax.set(title=f"{region} ({df.loc[df.region == region, 'crs'].iloc[0]})", xlabel="E (km)", ylabel="N (km)")
    ax.set_aspect("equal")
    ax.legend(markerscale=8)
plt.tight_layout()
plt.show()

## 4. Phân bố lớp
Đếm pixel từ `verification.json` (đã quét toàn bộ mẫu lúc thu). Thang log.

In [ ]:
pixels = pd.DataFrame({s: v["label_pixel_counts"] for s, v in verifications.items()}).fillna(0)
pixels.index = [CLASS_NAMES[int(i)] for i in pixels.index]
pixels = pixels.sort_values(pixels.columns[0])
ax = pixels.plot.barh(figsize=(10, 7), logx=True)
ax.set(title="Số pixel theo lớp", xlabel="pixel (log)")
plt.tight_layout()
plt.show()
display((pixels / pixels.sum() * 100).round(2).rename(columns=lambda c: c + " (%)"))

## 5. Mẫu ngẫu nhiên: ảnh RGB theo thời gian và nhãn
RGB = B4/B3/B2, chia 3000 và gamma 1/1.3. Nhãn xám là void. Khung trắng/xám mờ là mây (bộ dữ liệu không lọc mây).

In [ ]:
def rgb(image):
    return np.clip(image[[2, 1, 0]].transpose(1, 2, 0) / 3000, 0, 1) ** (1 / 1.3)


def show_sample(season, path, max_dates=6):
    sample = load_sample(season, path)
    image, labels, meta = sample["img"], sample["labels"][0], sample["metadata"]
    # Collector orders frames by day-of-year, not by date; show them chronologically.
    chronological = np.argsort(meta["dates"])
    shown = chronological[np.linspace(0, len(image) - 1, min(len(image), max_dates), dtype=int)]
    fig, axes = plt.subplots(1, len(shown) + 1, figsize=(2.2 * (len(shown) + 1), 2.6))
    for ax, t in zip(axes, shown):
        ax.imshow(rgb(image[t]))
        ax.set_title(meta["dates"][t], fontsize=8)
    axes[-1].imshow(labels, cmap=LABEL_CMAP, vmin=0, vmax=19, interpolation="nearest")
    axes[-1].set_title("labels[0]", fontsize=8)
    for ax in axes:
        ax.axis("off")
    present = [CLASS_NAMES[c] for c in np.unique(labels)]
    fig.suptitle(f"{season} | {meta['region']} | {meta['id']} | T={len(image)} | " + ", ".join(present),
                 fontsize=8)
    plt.tight_layout()
    plt.show()


rng = np.random.default_rng(SEED)
for season, part in df.groupby("season"):
    for path in rng.choice(part["path"].to_numpy(), size=min(SAMPLES_PER_SEASON, len(part)), replace=False):
        show_sample(season, path)

## 6. Đường NDVI trung bình theo lớp
NDVI = (B8 − B4)/(B8 + B4), chỉ pixel hợp lệ (`valid_mask`) và có nhãn cây trồng; gom theo tháng của mùa (Oct → Sep).

In [ ]:
SEASON_MONTHS = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
fig, axes = plt.subplots(1, len(SEASONS), figsize=(7 * len(SEASONS), 4.5), squeeze=False, sharey=True)
for ax, (season, part) in zip(axes[0], df.groupby("season")):
    sums = np.zeros((20, 12))
    counts = np.zeros((20, 12))
    paths = rng.choice(part["path"].to_numpy(), size=min(PROFILE_SAMPLES, len(part)), replace=False)
    for path in paths:
        sample = load_sample(season, path)
        image, labels, valid = sample["img"], sample["labels"][0], sample["valid_mask"]
        red, nir = image[:, 2], image[:, 6]
        ndvi = (nir - red) / np.maximum(nir + red, 1)
        for t, day in enumerate(sample["metadata"]["dates"]):
            m = SEASON_MONTHS.index(int(day[5:7]))
            ok = valid[t] & (labels != VOID) & (labels != 0)
            np.add.at(sums[:, m], labels[ok], ndvi[t][ok])
            np.add.at(counts[:, m], labels[ok], 1)
    top = np.argsort(counts.sum(axis=1))[::-1][:8]
    for c in top[counts[top].sum(axis=1) > 0]:
        mean = np.where(counts[c] > 0, sums[c] / np.maximum(counts[c], 1), np.nan)
        ax.plot(range(12), mean, marker="o", label=CLASS_NAMES[c], color=LABEL_CMAP(c))
    ax.set_xticks(range(12), SEASON_MONTHS)
    ax.set(title=f"{season} ({len(paths)} mẫu)", xlabel="tháng", ylabel="NDVI trung bình")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 7. Phổ trung bình theo lớp
Reflectance ×10000 trung bình trên pixel hợp lệ của mọi ngày, 200 mẫu đầu manifest mỗi mùa.

In [ ]:
fig, axes = plt.subplots(1, len(SEASONS), figsize=(7 * len(SEASONS), 4.5), squeeze=False, sharey=True)
for ax, (season, part) in zip(axes[0], df.groupby("season")):
    sums, counts = np.zeros((20, 10)), np.zeros(20)
    for path in part["path"].to_numpy()[:200]:
        sample = load_sample(season, path)
        image, labels, valid = sample["img"], sample["labels"][0], sample["valid_mask"]
        for t in range(len(image)):
            ok = valid[t] & (labels != VOID)
            np.add.at(sums, labels[ok], image[t][:, ok].T)
            np.add.at(counts, labels[ok], 1)
    for c in np.argsort(counts)[::-1][:8]:
        if counts[c]:
            ax.plot(BANDS, sums[c] / counts[c], marker="o", label=CLASS_NAMES[c], color=LABEL_CMAP(c))
    ax.set(title=season, ylabel="reflectance ×10000")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()